Layers: PW supply · Clusters · TDS · Demand · SWD wells · Oil wells

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import branca.colormap as cm
from folium.plugins import MarkerCluster
import os

In [ ]:
N_COLAB = 'google.colab' in str(dir())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = "/content/drive/MyDrive/YOUR_FOLDER/pw_analysis"
else:
    WORK_DIR = "."

DATA_DIR = f"{WORK_DIR}/data_raw/NM geofiles"
OUT_DIR = f"{WORK_DIR}/outputs"

LOAD ALL DATA

In [ ]:
nm_counties = gpd.read_file(f"{DATA_DIR}/nm_counties.geojson")
nm_counties["county_key"] = (
    nm_counties["NAME"].str.strip().str.lower()
    .str.replace("doña ana", "dona ana", regex=False)
    .str.replace("debaca", "de baca", regex=False)
)
print(f"✓ Counties: {len(nm_counties)}")

In [ ]:
# 1b. PW supply + cross-validation (county level)
supply = pd.read_csv(f"{WORK_DIR}/data_clean/data_processed/nm_county_summary.csv")
supply["county_key"] = supply["county_key"].str.strip().str.lower()
print(f"✓ Supply data: {len(supply)} counties")

In [ ]:
# 1c. Demand (county level)
demand = pd.read_csv(f"{WORK_DIR}/data_processed/nm_water_demand_2025_final.csv")
demand["county_key"] = (demand["county_key"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ")
)
print(f"✓ Demand data: {len(demand)} counties")

In [ ]:
# 1d. PW clusters — for TDS (township level, aggregated to county)
#     Adjust filename if different in your Drive
nm_pw = pd.read_csv(f"{WORK_DIR}/data_clean/data_processed/nm_cross_validation.csv")
nm_pw["county_key"] = nm_pw["county"].str.strip().str.lower()
print(f"✓ PW clusters: {len(nm_pw)} townships")

In [ ]:
# 1e. Well point data
swd_wells = gpd.read_file(f"{WORK_DIR}/data_clean/data_processed/ocd/nm_ocd_disposal_wells.geojson")
oil_wells = gpd.read_file(f"{WORK_DIR}/data_clean/data_processed/ocd/nm_ocd_oil_wells.geojson")
oil_sample = oil_wells.sample(n=min(5000, len(oil_wells)), random_state=42)
print(f"✓ SWD wells: {len(swd_wells):,}")
print(f"✓ Oil sample: {len(oil_sample):,}")

In [ ]:
#1f. Drought
drought_raw = pd.read_csv(f"{WORK_DIR}/data_raw/nm_drought_2020_2025.csv")

In [ ]:
drought = (drought_raw
    .iloc[:, 6:10].copy()
    .dropna(subset=[drought_raw.columns[6]])
    .reset_index(drop=True)
)
drought.columns = ["county_raw","weeks_in_drought",
                   "total_weeks","pct_weeks_drought"]
drought = drought[drought["county_raw"].astype(str)
                  .str.contains("County", na=False)]
drought["pct_weeks_drought"] = pd.to_numeric(
    drought["pct_weeks_drought"], errors="coerce")
drought["county_key"] = (drought["county_raw"]
    .str.replace(" County","", regex=False)
    .str.strip().str.lower()
    .str.replace("_"," ")
    .str.replace("debaca","de baca")
    #.str.replace("dona ana","doña ana")
)

In [ ]:
print(drought[drought["county_raw"].str.contains("Dona", na=False)]
      [["county_raw","county_key"]].to_string(index=False))

print(nm_map[nm_map["NAME"].str.contains("Ana", na=False)]
      [["NAME","county_key","pct_weeks_drought"]].to_string(index=False))

In [ ]:
#drought["county_key"] = drought["county_key"].replace(
  #  "dona ana", "doña ana"
#)

In [ ]:
print(f"✓ Counties:  {len(nm_counties)}")
print(f"✓ Supply:    {len(supply)} counties")
print(f"✓ Demand:    {len(demand)} counties")
print(f"✓ Drought:   {len(drought)} counties")
print(f"✓ SWD wells: {len(swd_wells):,}")

COUNTY-LEVEL GEODATAFRAME

In [ ]:
#2a. TDS — aggregate from township to county
county_tds = (nm_pw
    .groupby("county_key")
    .agg(
        mean_tds_mgl   = ("tds_real", "mean"),
        median_tds_mgl = ("tds_real", "median"),
        min_tds_mgl    = ("tds_real", "min"),
        max_tds_mgl    = ("tds_real", "max"),
    )
    .round(1)
    .reset_index()
)


nm_map = (nm_counties
    .merge(supply[["county_key","total_pw_vol",
                   "dominant_cluster","swd_well_count"]],
           on="county_key", how="left")
    .merge(demand[["county_key","agri_tpw_bbl_2025",
                   "power_tw_bbl_2025","total_demand_bbl_2025",
                   "plant_count"]],
           on="county_key", how="left")
    .merge(county_tds[["county_key","mean_tds_mgl"]],
           on="county_key", how="left")
    .merge(drought[["county_key","pct_weeks_drought",
                    "weeks_in_drought"]],
           on="county_key", how="left")
)

for col in ["total_pw_vol","swd_well_count","agri_tpw_bbl_2025",
            "power_tw_bbl_2025","total_demand_bbl_2025",
            "plant_count","mean_tds_mgl"]:
    nm_map[col] = nm_map[col].fillna(0)

nm_map["dominant_cluster"] = nm_map["dominant_cluster"].fillna(-1)

print(f"✓ nm_map: {len(nm_map)} counties")
print(f"  PW data:  {(nm_map['total_pw_vol']>0).sum()}")
print(f"  Demand:   {(nm_map['total_demand_bbl_2025']>0).sum()}")
print(f"  Drought:  {nm_map['pct_weeks_drought'].notna().sum()}")

In [ ]:
# Replace exact zeros with NaN → white on map
for col in ["total_pw_vol","mean_tds_mgl","total_demand_bbl_2025",
            "agri_tpw_bbl_2025","power_tw_bbl_2025"]:
    nm_map[col] = nm_map[col].replace(0, float("nan"))

# Log scale for PW volume
nm_map["pw_vol_log"] = np.log10(
    nm_map["total_pw_vol"].replace(0, float("nan"))
)

# Scaled columns — engineering units
nm_map["pw_vol_M"]  = nm_map["total_pw_vol"]          / 1e6  # M bbl
nm_map["demand_M"]  = nm_map["total_demand_bbl_2025"] / 1e6  # M bbl/yr
nm_map["agri_M"]    = nm_map["agri_tpw_bbl_2025"]    / 1e6  # M bbl/yr
nm_map["power_M"]   = nm_map["power_tw_bbl_2025"]     / 1e6  # M bbl/yr
nm_map["tds_k"]     = nm_map["mean_tds_mgl"]          / 1e3  # mg/L

# Annual supply
nm_map["annual_supply_bbl"] = nm_map["total_pw_vol"].fillna(0) / 5

# Formatted tooltip values — engineering units
def fmt_bbl(val):
    if pd.isna(val) or val == 0: return "No data"
    if val >= 1e9:  return f"{val/1e9:.2f}B bbl"
    if val >= 1e6:  return f"{val/1e6:.1f}M bbl"
    if val >= 1e3:  return f"{val/1e3:.1f}K bbl"
    return f"{val:.0f} bbl"

nm_map["supply_fmt"]  = nm_map["annual_supply_bbl"].apply(
    lambda x: fmt_bbl(x)+"/yr" if x > 0 else "No data")
nm_map["pw_vol_fmt"]  = nm_map["total_pw_vol"].apply(
    lambda x: fmt_bbl(x)+" (5yr)")
nm_map["demand_fmt"]  = nm_map["total_demand_bbl_2025"].apply(
    lambda x: fmt_bbl(x)+"/yr" if pd.notna(x) else "No data")
nm_map["agri_fmt"]    = nm_map["agri_tpw_bbl_2025"].apply(
    lambda x: fmt_bbl(x)+"/yr" if pd.notna(x) else "No data")
nm_map["power_fmt"]   = nm_map["power_tw_bbl_2025"].apply(
    lambda x: fmt_bbl(x)+"/yr" if pd.notna(x) else "No data")
nm_map["tds_fmt"]     = nm_map["mean_tds_mgl"].apply(
    lambda x: f"{x/1e3:.1f} mg/L" if pd.notna(x) else "No data")
nm_map["drought_fmt"] = nm_map["pct_weeks_drought"].apply(
    lambda x: f"{x:.1f}%" if pd.notna(x) else "No data")

print("✓ All columns ready")

------------------------------------

map build

In [ ]:
CLUSTER_COLORS = {-1:"#F1EFE8", 0:"#93C5FD", 1:"#34D399",
                   2:"#FBBF24", 3:"#EF4444"}

In [ ]:
tds_cmap = cm.LinearColormap(
    colors=["#FFF0F5","#FECDD3","#FDA4AF","#F43F5E","#881337"],
    vmin=float(nm_map["tds_k"].min()),
    vmax=float(nm_map["tds_k"].max()),
    caption="Mean TDS (mg/L)" # not k mg/L
)

def tds_style(feature):
    val = feature["properties"].get("tds_k")
    if val is None or val != val or val == 0:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}
    try:
        return {"fillColor":  tds_cmap(float(val)),
                "color":      "#CBD5E1",
                "weight":     0.5,
                "fillOpacity":0.70}
    except:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}

In [ ]:
power_cmap = cm.LinearColormap(
    colors=["#FFF7ED","#FED7AA","#FB923C","#EA580C","#7C2D12"],
    vmin=float(nm_map["power_M"].dropna().min()),
    vmax=float(nm_map["power_M"].dropna().max()),
    caption="Power demand 2025 (M bbl/yr)"
)

def power_style(feature):
    val = feature["properties"].get("power_M")
    if val is None or val != val or val == 0:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}
    try:
        return {"fillColor":  power_cmap(float(val)),
                "color":      "#CBD5E1",
                "weight":     0.5,
                "fillOpacity":0.70}
    except:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}

In [ ]:
m = folium.Map(location=[34.5,-106.0], zoom_start=7,
               tiles="CartoDB positron")

# Layer 1: PW volume (log scale)
folium.Choropleth(
    geo_data=nm_map.to_json(), data=nm_map,
    columns=["county_key","pw_vol_log"],
    key_on="feature.properties.county_key",
    fill_color="Blues", fill_opacity=0.65, line_opacity=0.3,
    nan_fill_color="#FFFFFF", nan_fill_opacity=0.4,
    legend_name="PW volume — log₁₀(bbl)",
    name="PW volume (supply)", show=True
).add_to(m)

# Layer 2: PW clusters
folium.GeoJson(
    nm_map, name="PW clusters",
    style_function=lambda f: {
        "fillColor": CLUSTER_COLORS.get(
            int(f["properties"].get("dominant_cluster",-1)),"#F1EFE8"),
        "color":"white","weight":0.8,"fillOpacity":0.65},
    show=False
).add_to(m)

# Layer 3: TDS (k mg/L)
#folium.Choropleth(
    #geo_data=nm_map.to_json(), data=nm_map,
    #columns=["county_key","tds_k"],
    #key_on="feature.properties.county_key",
    #fill_color="RdYlGn_r", fill_opacity=0.70, line_opacity=0.2,
    #nan_fill_color="#FFFFFF", nan_fill_opacity=0.4,
    #legend_name="Mean TDS (k mg/L)",
    #name="TDS salinity", show=False
#).add_to(m)

# Layer 4: Total demand (M bbl/yr)
folium.Choropleth(
    geo_data=nm_map.to_json(), data=nm_map,
    columns=["county_key","demand_M"],
    key_on="feature.properties.county_key",
    fill_color="YlOrRd", fill_opacity=0.65, line_opacity=0.2,
    nan_fill_color="#FFFFFF", nan_fill_opacity=0.4,
    legend_name="Total demand 2025 (M bbl/yr)",
    name="Total demand 2025", show=False
).add_to(m)

# Layer 5: Irrigation (M bbl/yr)
folium.Choropleth(
    geo_data=nm_map.to_json(), data=nm_map,
    columns=["county_key","agri_M"],
    key_on="feature.properties.county_key",
    fill_color="Greens", fill_opacity=0.65, line_opacity=0.2,
    nan_fill_color="#FFFFFF", nan_fill_opacity=0.4,
    legend_name="Irrigation demand 2025 (M bbl/yr)",
    name="Irrigation demand 2025", show=False
).add_to(m)

# Layer 6: Power (M bbl/yr)
#folium.Choropleth(
   # geo_data=nm_map.to_json(), data=nm_map,
    #columns=["county_key","power_M"],
    #key_on="feature.properties.county_key",
    #fill_color="Oranges", fill_opacity=0.65, line_opacity=0.2,
    #nan_fill_color="#FFFFFF", nan_fill_opacity=0.4,
    #legend_name="Power demand 2025 (M bbl/yr)",
   # name="Power demand 2025", show=False
#).add_to(m)

In [ ]:
#layer 3
folium.GeoJson(
    nm_map,
    name="TDS salinity",
    style_function=tds_style,
    show=False,
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME","tds_fmt"],
        aliases=["County:","Mean TDS:"],
        localize=False
    )
).add_to(m)

tds_cmap.add_to(m)

In [ ]:
#layer 6 Power (M bbl/yr)
folium.GeoJson(
    nm_map,
    name="Power demand 2025",
    style_function=power_style,
    show=False,
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME","power_fmt"],
        aliases=["County:","Power demand 2025:"],
        localize=False
    )
).add_to(m)
power_cmap.add_to(m)

In [ ]:
drought_cmap = cm.LinearColormap(
    colors=["#FFF5F5","#FECACA","#EF4444","#B91C1C","#7F1D1D"],
    vmin=0, vmax=100,
    caption="% of weeks in drought 2020–2025 (USDM)"
)

def drought_style(feature):
    val = feature["properties"].get("pct_weeks_drought")
    if val is None or val != val:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}
    try:
        return {"fillColor":  drought_cmap(float(val)),
                "color":      "#CBD5E1",
                "weight":     0.5,
                "fillOpacity":0.72}
    except:
        return {"fillColor":"#FFFFFF","color":"#CBD5E1",
                "weight":0.5,"fillOpacity":0.1}

folium.GeoJson(
    nm_map,
    name="Drought severity 2020–2025",
    style_function=drought_style,
    show=False,
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME","drought_fmt","weeks_in_drought"],
        aliases=["County:","Drought % (2020–2025):",
                 "Weeks in drought:"],
        localize=False
    )
).add_to(m)
drought_cmap.add_to(m)

In [ ]:
# County tooltip
folium.GeoJson(
    nm_map, name="County labels",
    style_function=lambda x: {"fillOpacity":0,"weight":0},
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME","supply_fmt","pw_vol_fmt",
                "dominant_cluster","swd_well_count",
                "tds_fmt","demand_fmt","agri_fmt",
                "power_fmt","drought_fmt"],
        aliases=["County:","PW supply (annual):",
                 "PW vol (5yr):","Cluster:","SWD wells:",
                 "Mean TDS:","Total demand 2025:",
                 "Irrigation 2025:","Power 2025:",
                 "Drought (2020–2025):"],
        localize=False
    ), show=True
).add_to(m)

# SWD wells
swd_group = folium.FeatureGroup(name="SWD wells", show=True)
for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#059669", fill=True,
        fill_color="#1D9E75", fill_opacity=0.8, weight=0.5,
        popup=folium.Popup(
            f"<b>{row.get('name','SWD Well')}</b><br>"
            f"County: {str(row.get('county','')).title()}",
            max_width=200)
    ).add_to(swd_group)
swd_group.add_to(m)

# Oil wells
oil_cluster = MarkerCluster(name="Oil wells (5k sample)", show=False)
for _, row in oil_sample.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2, color="#64748B", fill=True,
        fill_color="#888780", fill_opacity=0.5, weight=0
    ).add_to(oil_cluster)
oil_cluster.add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;right:10px;z-index:1000;
  background:white;padding:14px 18px;border-radius:10px;
  border:1px solid #CBD5E1;font-family:sans-serif;font-size:11px;
  box-shadow:0 2px 10px rgba(0,0,0,0.12);min-width:220px">
  <b style="font-size:13px">NM Produced Water Map</b><br>
  <span style="color:#64748B;font-size:10px">
    Cross-validation: Spearman r = 0.900</span><br><br>
  <b style="color:#475569;font-size:10px">SUPPLY</b><br>
  <span style="background:#2563EB;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;PW volume (log₁₀ bbl)<br>
  <span style="background:#FBBF24;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;PW clusters<br>
  <span style="background:#F43F5E;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;TDS salinity ( mg/L)<br><br>
  <b style="color:#475569;font-size:10px">DEMAND</b><br>
  <span style="background:#FC8D59;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;Total demand (M bbl/yr)<br>
  <span style="background:#74C476;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;Irrigation (M bbl/yr)<br><br>
  <b style="color:#475569;font-size:10px">DROUGHT</b><br>
  <span style="background:#EF4444;display:inline-block;width:12px;
    height:12px;opacity:.65;border-radius:2px;
    vertical-align:middle"></span>&nbsp;% weeks drought 2020–25<br><br>
  <b style="color:#475569;font-size:10px">INFRASTRUCTURE</b><br>
  <span style="background:#1D9E75;display:inline-block;width:9px;
    height:9px;border-radius:50%;vertical-align:middle"></span>
  &nbsp;SWD wells<br>
  <span style="background:#888780;display:inline-block;width:9px;
    height:9px;border-radius:50%;vertical-align:middle"></span>
  &nbsp;Oil wells (sample)
</div>"""

m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=True).add_to(m)

OUTPUT_FILE = f"{OUT_DIR}/nm_combined_map5.html"
m.save(OUTPUT_FILE)
print(f"✓ Saved: nm_combined_map5.html")

from google.colab import files
files.download(OUTPUT_FILE)
m